# SCA13 — Final MNTB model
Macica et al. (2003), Figures 9, 1d–f and 8. Model A: g_LT=27 nS, g_Na=510 nS, Wang h kinetics. Figure 9 is calibration; Figures 1 and 8 are experimental comparisons, not exact numerical replicas. Keep the NESTML file beside this notebook. Run in ataxia_env. No parameter sweeps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import nest
from pathlib import Path
from pynestml.frontend.pynestml_frontend import generate_nest_target
source = Path("hh_neuron_sca13_final.nestml").resolve()
assert source.is_file(), source
module_name = "sca13_final_module"
model = "hh_neuron_sca13_final_nestml"
target = Path("target_final").resolve()
generate_nest_target(input_path=str(source), target_path=str(target), module_name=module_name, suffix="_nestml")
module = target / (module_name + ".so")
assert module.is_file(), module
nest.ResetKernel()
if model not in nest.node_models:
    nest.Install(str(module))
for k,v in {"g_LT":27., "g_Na":510., "g_L":2., "C_m":10., "E_L":-63.}.items():
    assert np.isclose(nest.GetDefaults(model)[k], v), k
results_dir = Path("final_model_results")
results_dir.mkdir(exist_ok=True)
DT, T0, AMP, DUR, AP_THRESHOLD = 0.005, 50., 1400., 0.25, -10.
# Preserve the 0.01 ms stimulus rounding of the distributed sensitivity notebook.
# If your local calibration used DT-grid rounding instead, change this flag.
GRID_ROUNDING = False


In [ ]:
def round_time(t):
    return round(round(t / DT) * DT, 6) if GRID_ROUNDING else round(t, 2)

def run_train(g_kv31, freq, n_pulses, amp=AMP, dur=DUR):
    nest.ResetKernel()
    nest.set_verbosity("M_ERROR")
    nest.SetKernelStatus({"resolution": DT})
    if model not in nest.node_models:
        nest.Install(str(module))
    neuron = nest.Create(model, params={"g_Kv31": g_kv31})
    mm = nest.Create("multimeter", params={"record_from": ["V_m"], "interval": DT})
    nest.Connect(mm, neuron)

    period = 1000.0 / freq
    starts = [round_time(T0 + k * period) for k in range(n_pulses)]
    times = []
    values = []
    for s in starts:
        times += [s, round_time(s + dur)]
        values += [amp, 0.0]
    gen = nest.Create("step_current_generator",
                      params={"amplitude_times": times, "amplitude_values": values})
    nest.Connect(gen, neuron, syn_spec={"delay": DT})
    nest.Simulate(round_time(starts[-1] + 40.0))

    ev = mm.get("events")
    t = np.array(ev["times"])
    V = np.array(ev["V_m"])
    peaks = []
    delays = []
    for i, s in enumerate(starts):
        end = starts[i+1] if i+1 < len(starts) else s + period
        window = (t >= s + DT) & (t < end + DT)
        k = np.argmax(V[window])
        peaks.append(V[window][k])
        delays.append(t[window][k] - s)
    peaks = np.array(peaks)
    return {"t": t, "V": V, "starts": np.array(starts),
            "peaks": peaks, "delay": np.array(delays), "ok": peaks >= AP_THRESHOLD}


def plot_delay(ax,r):
    ok=r['ok']; x=np.arange(len(ok)); y=r['delay'].copy(); y[~ok]=np.nan
    ax.plot(x,y,'k.-',ms=3)
    ax.plot(x[~ok],np.full(np.sum(~ok),0.3),'k',marker=r'$\uparrow$',ls='none')
    ax.text(.98,.95,f'{np.sum(~ok)} failures',transform=ax.transAxes,ha='right',va='top')
    ax.set_ylim(0,max(1.5,1.08*np.max(y[ok]) if np.any(ok) else 1.5))

def plot_aps(ax,r,f):
    for i,s in enumerate(r['starts']):
        if not r['ok'][i]: continue
        end=r['starts'][i+1] if i+1<len(r['starts']) else s+1000/f
        w=(r['t']>=s-.2)&(r['t']<end)&(r['t']<=s+2.)
        ax.plot(r['t'][w]-s,r['V'][w],color='black',alpha=.45,lw=.8)
    ax.set(xlim=(-.2,2),ylim=(-90,60))
    ax.text(.98,.95,f"{sum(r['ok'])}/{len(r['ok'])} APs",transform=ax.transAxes,ha='right',va='top')

def metrics(results):
    rows=[]
    for (cond,f),r in results.items():
        ok=r['ok']; d=r['delay'][ok]; p=r['peaks'][ok]
        rows.append(dict(Condition=cond,Frequency=f,Failures=int(sum(~ok)),Reliability=100*np.mean(ok),Mean_delay=np.mean(d) if len(d) else np.nan,Jitter=np.std(d) if len(d) else np.nan,Mean_peak=np.mean(p) if len(p) else np.nan))
    return pd.DataFrame(rows)

def save(name,fig,results):
    fig.savefig(results_dir/f'{name}.png',dpi=300,bbox_inches='tight')
    df=metrics(results); df.to_csv(results_dir/f'{name}.csv',index=False)
    with open(results_dir/f'{name}.pkl','wb') as h: pickle.dump(results,h)
    display(df.round(3))
    plt.show()

def combined_plot(results,conditions,fa,fb,title):
    fig,axes=plt.subplots(len(fa),4,figsize=(16,3*len(fa)))
    for i,(f1,f2) in enumerate(zip(fa,fb)):
        for j,cond in enumerate(conditions):
            a,b=axes[i,j],axes[i,j+2]
            plot_delay(a,results[cond,f1]); plot_aps(b,results[cond,f2],f2)
            if j==0:
                a.set_ylabel(f'{f1} Hz\nDelay (ms)'); b.set_ylabel(f'{f2} Hz\nV (mV)')
            if i==0:
                a.set_title('a — '+cond); b.set_title('b — '+cond)
            if i==len(fa)-1:
                a.set_xlabel('Stimulus number'); b.set_xlabel('Time from stimulus (ms)')
    ymax=max(ax.get_ylim()[1] for ax in axes[:,:2].flat)
    for ax in axes[:,:2].flat: ax.set_ylim(0,ymax)
    fig.suptitle(title); fig.tight_layout(rect=[0,0,1,.96]); return fig


## Figure 9 — Computational model
1.4 nA, 0.25 ms; 150 vs 100 nS; nominal 100 ms trains (analysis choice). Failure threshold −10 mV. Expected calibrated failures at 350/360/410 Hz: control 2/5/13, reduced 0/0/13. A mismatch is reported, not silently tuned.

In [ ]:
g9={'Control (150 nS)':150.,'Reduced (100 nS)':100.}
r9={(cond,f):run_train(g,f,int(f/10)) for cond,g in g9.items() for f in [100,300,350,360,410]}
for cond,expected in zip(g9,[[2,5,13],[0,0,13]]):
    actual=[int(sum(~r9[cond,f]['ok'])) for f in [350,360,410]]
    print(cond,actual,'expected',expected)
    if actual!=expected: print('WARNING: compare timing convention with local calibration before interpreting.')
fig9=combined_plot(r9,g9,[300,350,360,410],[100,350,360,410],'Figure 9 — Model A')
save('Figure9',fig9,r9)

## Figure 1d–f — WT, TEA and knockout
Experimental pulses: 2 nA, 0.3 ms, 100–300 Hz. Nominal 100 ms trains. TEA 27 nS and KO 15 nS are effective residual high-threshold conductance approximations, not measured absolute conductances. KO residual current is not literal Kv3.1 expression; compensation and Kv3.3 are not explicitly modeled.

In [ ]:
g1={'WT (150 nS)':150.,'TEA (27 nS)':27.,'KO proxy (15 nS)':15.}
r1={(cond,f):run_train(g,f,int(f/10),amp=2000.,dur=.3) for cond,g in g1.items() for f in [100,200,300]}
fig1,axes=plt.subplots(3,3,figsize=(15,9),sharex=True,sharey=True)
for i,f in enumerate([100,200,300]):
    for j,cond in enumerate(g1):
        ax=axes[i,j]; r=r1[cond,f]; t=r['t']-r['starts'][0]; w=(t>=-2)&(t<=100)
        ax.plot(t[w],r['V'][w],'k',lw=.8)
        ax.plot(r['starts'][~r['ok']]-r['starts'][0],np.full(sum(~r['ok']),-85),'k|',ms=7)
        ax.text(.98,.95,f"{sum(r['ok'])}/{len(r['ok'])} APs",transform=ax.transAxes,ha='right',va='top')
        if i==0: ax.set_title(cond)
        if j==0: ax.set_ylabel(f'{f} Hz\nV (mV)')
        if i==2: ax.set_xlabel('Time (ms)')
        ax.set_ylim(-90,60)
fig1.suptitle('Figure 1d–f — Model A'); fig1.tight_layout(rect=[0,0,1,.96]); save('Figure1',fig1,r1)

## Figure 8 — Control vs PMA
Experimental comparison at 100, 300, 400 Hz. PMA proxy 83 nS approximates the reported 44.8% reduction in I_HT. Pulse amplitude/duration assumed equal to Figure 1 (2 nA, 0.3 ms); pulse counts 11, 17, 25 are approximate readings adopted from the previous comparison notebook. Figure 8 is a single-cell experimental pattern, not an exact model target.

In [ ]:
g8={'Control (150 nS)':150.,'PMA proxy (83 nS)':83.}
n8={100:11,300:17,400:25}
r8={(cond,f):run_train(g,f,n8[f],amp=2000.,dur=.3) for cond,g in g8.items() for f in n8}
fig8=combined_plot(r8,g8,list(n8),list(n8),'Figure 8 — Model A')
save('Figure8',fig8,r8)

## Interpretation
Compare WT/TEA/KO frequency-dependent loss of firing, and PMA effects on reliability and latency dispersion. Report mismatches without retuning. Figure 9 was used to select Model A; Figures 1 and 8 are additional checks. Reference: Macica et al., J Neurosci 23(4), 1133–1141 (2003).